# Custom Climate Profiles Generation

<br> Many energy system modeling and other planning processes are designed to intake hourly information in order to capture short term fluctuations in conditions that affect energy supply and demand. An *Annualized Hourly Climate Profile* is a dataset that contains hourly weather conditions at a given location for an entire year, and is commonly referred to as an **8760 Climate Profile**. Climate profiles can represent future weather conditions in a changing climate and can thus better inform design and planning processes for a wide range of future needs. Guidance, methodologies, and recommendations for <span style="color: blue;\">[Cal-Adapt Climate Profiles](https://analytics.cal-adapt.org/climate-profiles/)</span> is available for reference. A comprehensive set of pre-generated Climate Profiles for the most common use-cases is available via the <span style="color: blue;\">[Cal-Adapt: Data Download Tool](https://cal-adapt.org/dashboard/data-download-tool)</span>.

Cal-Adapt provides three kinds of Climate Profiles: 
1. **Standard Year 8760**:  One year of hourly data that represents any desired statistical percentile of weather conditions for a location over a 30-year climatological period. A Standard Year builds a climate profile for each climate model on a single variable and can be used to evaluate both median and extreme conditions by selecting appropriate percentiles. Both historical and future Standard Year 8760s are available.

2. **Typical Meteorological Year 8760 (TMY)**: One year of hourly data that represents the median weather conditions for a location over a climatological period. A TMY is built from ten specific weather variables that are weighted in compliance with <span style="color: blue;\">[TMY standard methodology](https://nsrdb.nrel.gov/data-sets/tmy)</span>. TMYs statistically assess the median conditions from 30 years of model data and select the most “typical” month for each month during a year. These are then compiled into a single climate profile. The resulting TMY file includes multiple variables and has very specific formatting requirements. Both historical and future TMY (FTMY) 8760s are available.

3. **Extreme Meteorological Year 8760 (XMY)**: One year of hourly data that represents extreme weather conditions for a location over a climatological period. XMYs come in two different extremes types:
    - *Shock Event XMY*: An extreme climate profile that is representative of short-term, event-wise pulses that increase peak demand, such as a 4-day heatwave or a 5-day coldsnap. The Shock XMY is built upon an extreme cumulative distribution furthest from the median weather conditions for a location over a reference period. 
    - *Persistence XMY*: An extreme climate profile that is representative of extreme annual metrics, such as an extreme hot year or an extreme cold year, that can be applied for highly tailorable system stress tests. The Persistence XMY is built upon a given statistical extreme percentile of weather conditions for a location over a reference period.


**Intended Application**: As a user, I want to <span style="color:#FF0000">**generate a custom climate profile**</span> for my area:
1. Standard Year 8760 profile, with my variable and extreme conditions of choice.
2. TMY 8760 profile, for the closest weather station to my area.
3. Hot Extreme Shock XMY profile, for my asset location. 

#### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.explore.extreme_meteorological_year import (
    shock_XMY, 
    persistence_XMY,
)
from climakitae.core.data_interface import get_subsetting_options

import warnings
warnings.filterwarnings("ignore")

### Generate a Custom Standard Year Profile 

AE makes it easy to generate a custom Standard Year climate profile for you. Expand the **Customization Guide** for available options.
<details>
<summary><strong><span style="color: #1577b4;">▶ Click Here for the Standard Year Customization Guide</span></strong></summary>

---
|**Argument**|**Options**|**Notes**|
|------------|-----------|-----------------------|
|variable|WRF variables and units| Check out our variable and units list <span style="color: blue;\">[here](https://github.com/cal-adapt/climakitae/blob/main/climakitae/data/variable_descriptions.csv)</span>.|
|units| WRF variables and units|Check out our variable and units list <span style="color: blue;\">[here](https://github.com/cal-adapt/climakitae/blob/main/climakitae/data/variable_descriptions.csv)</span>.|
|qtile|0-1|Statistical quantile to generate Standard year profile; median = 0.5|
|no_delta|True, False |Option to take a delta signal difference between your selected GWL and the historical reference GWL 1.2°C. Default = False|
|resolution|"3 km", "9 km", "45 km"| Spatial resolution of input data. Default = "3 km"|
|location |HadISD weather stations, lat-lon pair, cached area|Check out the <span style="color: blue;\">[available station list](https://cadcat.s3.amazonaws.com/hadisd/hadisd_stations.csv)</span> for station options. For the full list of cached areas available on AE, the `get_subsetting_options` function can help. Check out the <span style="color: blue;\">[basic_data_access notebook](https://github.com/cal-adapt/cae-notebooks/blob/main/data-access/basic_data_access.ipynb)</span> for how to use this function.  Note: there is a small buffer applied to lat-lon arguments to retrieve the closest gridcell. |
|approach|"Warming Level", "Time"|Default = "Warming Level"|
|warming_level|Valid range is 0.5 - 4.0°C|Fully customizeable GWL. Default GWL = 1.2°C |
|warming_level_window|Valid range is 5-25 years|Warming level window size. Default = 15 years (30-year window)|
|centered_year|Valid range is 1980 - 2099| Required for time-based approach. |
|time_profile_scenario|"SSP2-4.5", "SSP3-7.0", "SSP5-8.5"| Option to return a specified SSP scenario for time-based approach. Default="SSP3.7-0". Note: "SSP 2-4.5", "SSP 5-8.5" only valid if spatial resolution is "9 km", "45 km" and does not include bias-adjusted models |
|bias_adjusted_models| True, False| Option to return only bias-adjusted models. Default=False|

---
   
</details>

Now, set-up the profile generator -- the `get_climate_profile` function does all the work for you. Modify any argument based on your desired custom profile, options for each argument are listed in the table above!

In [2]:
# Set up the Standard Year generator
profile_selections = {
    ## Required variable and profile arguments
    "variable": "Air Temperature at 2m",
    "resolution": "3 km",
    "q": 0.5,
    "units": "degF",

    ## Required approach arguments, Options: "Warming Level", "Time"
    "approach": "Warming Level",
    "warming_level": [2.0], # GWL-option only
    #"centered_year": 2010, # Time-based option only

    ## Required location argument -- Uncomment your desired location option and modify
    "location": "Riverside Municipal Airport (KRAL)", # HadISD weather station
    # "location": (38.421, -121.41), # latitude and longitude
    # "location": "Orange County", # cached area

    ## Additional optional arguments -- Uncomment any desired options and modify
    "no_delta": True, 
    "warming_level_window": 5,
    # "time_profile_scenario": "SSP 2-4.5",
    # "bias_adjusted_models": True,
}

# Generate the climate profile
profile = get_climate_profile(**profile_selections)

📊 Retrieving climate data...


Data retrieval:   0%|          | 0/1 [00:00<?, ?dataset/s]

   📍 Converting 1 station(s) to lat/lon coordinates with ±0.02° buffer
      Latitude range: 33.9328 to 33.9728
      Longitude range: -117.4552 to -117.4152
Step 2: Data is being retrieved for warming level [2.0] across all available simulations and scenarios.
get data one var <xarray.DataArray np.str_('Air Temperature at 2m') (scenario: 1, simulation: 8,
                                                    time: 1051896)> Size: 34MB
dask.array<add, shape=(1, 8, 1051896), dtype=float32, chunksize=(1, 1, 14487), chunktype=numpy.ndarray>
Coordinates:
  * scenario           (scenario) <U22 88B 'Historical + SSP 3-7.0'
  * simulation         (simulation) <U26 832B 'WRF_FGOALS-g3_r1i1p1f1' ... 'W...
  * time               (time) datetime64[ns] 8MB 1980-09-01 ... 2100-08-31T23...
    Lambert_Conformal  int64 8B 0
Attributes:
    variable_id:           t2
    extended_description:  Temperature of the air 2m above Earth's surface. T...
    units:                 degF
    data_type:            

      Computing profiles:   0%|          | 0/8 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (8760, 8)
         With index: None, columns: [None]
         Units: degF
   ✓ No baseline subtraction requested, returning raw future profile


In [3]:
profile

,CESM2-r11i1p1f1-ssp370,CNRM-ESM2-1-r1i1p1f2-ssp370,EC-Earth3-r1i1p1f1-ssp370,EC-Earth3-Veg-r1i1p1f1-ssp370,FGOALS-g3-r1i1p1f1-ssp370,MIROC6-r1i1p1f1-ssp370,MPI-ESM1-2-HR-r3i1p1f1-ssp370,TaiESM1-r1i1p1f1-ssp370
1,60.457939,63.333061,66.988373,56.596519,65.960663,66.101562,65.229523,69.933319
2,57.794415,60.331650,62.918549,56.232544,62.749798,64.448288,61.336571,68.008621
3,57.714981,58.657280,56.601357,55.317215,60.272655,62.169281,59.114147,58.711498
4,55.907295,56.374870,54.503185,52.447643,58.560326,60.666901,57.659500,56.575699
5,53.867737,55.737007,56.818390,55.344299,57.721573,59.778603,56.526924,55.010696
...,...,...,...,...,...,...,...,...
8756,61.779869,64.102654,62.037117,57.030258,66.084427,71.263542,62.383678,67.546097
8757,62.225090,63.252148,63.745323,57.794746,68.998215,72.275223,64.779022,69.828453
8758,64.835602,67.148613,64.995346,58.559830,67.346420,71.443497,66.841820,71.421860
8759,65.259674,66.503441,65.524506,66.000870,69.978966,70.684402,64.276077,71.938873


**Export**. The following cell will export your custom generated Standard Year profile!

In [4]:
# export function uses the previously defined profile selections to generate the output file name
export_profile_to_csv(profile, **profile_selections)

### Generate a Typical Meteorological Year (TMY)

AE makes it easy to generate a custom TMY climate profiles for you. You can generate both a historical and future profiles. Expand the **Customization Guide** for available options.

<details>
<summary><strong><span style="color: #1577b4;">▶ Click Here for the TMY Customization Guide</span></strong></summary>

---
#### Typical Meteorological Year
The TMY methodology here mirrors that of the NSRDB TMY3 methodology which heavily weights the solar radiation input data. Be aware that the final selection of "typical" months may not be typical for other variables. Because a TMY represents average rather than extreme conditions, an TMY profile is not suited for designing systems to meet the worst-case conditions occurring at a location.

|**Argument**|**Options**|**Notes**|
|------------|-----------|-----------------------|
|Location: stations|HadISD weather stations | Check out the <span style="color: blue;\">[available station list](https://cadcat.s3.amazonaws.com/hadisd/hadisd_stations.csv)</span> for station options. Cached areas include counties, demand forecast zones, or service territories.|
|Location: latitude, longitude|Custom lat-lon pairs| Note: there is a small buffer applied to lat-lon arguments to retrieve the closest gridcell. |
|approach|"Warming Level", "Time"|Default = "Warming Level"|
|warming_level|Valid range is 0.5 - 4.0°C|Fully customizeable GWL. Default GWL = 1.2°C |
|start_year|Valid range is 1980-2099. | We recommend a 30-year period such as 1990-2020. |
|end_year|Valid range is 1980-2099.  | We recommend a 30-year period such as 1990-2020. |
|reanalysis|True, False  | True to use ERA5 reanalysis instead of models. Cannot be used with warming levels, and valid start and end years must be between 1981 and 2019.  |

---

</details>
 

In [ ]:
# Set up the TMY profile generator! The verbose option will output progress of the TMY generation
tmy = TMY(
    ## Location -- uncomment your desired option
    station_name = "Sacramento Executive Airport (KSAC)",
    # latitude = latitude,
    # longitude = longitude,

    # Approach -- uncomment your desired option
    warming_level = 2.0,
    # start_year = start_year,
    # end_year = end_year,

    ## Uncomment to use ERA5 reanalysis, valid only with time-based approach with start and end years between 1981 and 2019
    # reanalysis = True,
    verbose = True,
)

# Generate the profile!
tmy.generate_tmy()

**Export to non-EPW format.** TMY profiles are exported in `.epw` format by default, but can be exported as both `.csv` and `.tmy` file formats using the method `export_tmy_data` with the argument `extension="your_choice"` as shown below.

In [ ]:
tmy.export_tmy_data(extension="csv")

### Generate an Extreme Meteorological Year (XMY)

AE makes it easy to generate a custom XMY climate profiles for you. You can generate both a historical and future profiles. Expand the **Customization Guide** under each XMY type header for available options.

#### Shock Extreme Meteorological Year (shock XMY)

<details>
<summary><strong><span style="color: #1577b4;">▶ Click Here for the shock XMY Customization Guide</span></strong></summary>

---

The shock XMY methodology utilizes the following changes to the TMY methodology to capture cold and hot extremes:

1. Utilizes only air temperature, reflecting utility priorities when in using XMYs.
2. Calculates the relative difference between either the minimum (for cold shock) or maximum (for hot shock) air temperaure values from the median climatological trend.
3. For each month and simulation, selects the year in which air temperature was furthest from the climatological trend.


|**Argument**|**Options**|**Notes**|
|------------|-----------|-----------------------|
|extreme | "hot", "cold" | Type of shock extreme
|Location: station_name|HadISD weather stations | Check out the <span style="color: blue;\">[available station list](https://cadcat.s3.amazonaws.com/hadisd/hadisd_stations.csv)</span> for station options. Cached areas include counties, demand forecast zones, or service territories.|
|Location: latitude, longitude|Custom lat-lon pairs| Note: there is a small buffer applied to lat-lon arguments to retrieve the closest gridcell. |
|approach|"Warming Level", "Time"|Default = "Warming Level"|
|warming_level|Valid range is 0.5 - 4.0°C|Fully customizeable GWL. Default GWL = 1.2°C |
|start_year|Valid range is 1980-2099. | We recommend a 30-year period such as 1990-2020. |
|end_year|Valid range is 1980-2099.  | We recommend a 30-year period such as 1990-2020. |
|reanalysis|True, False  | True to use ERA5 reanalysis instead of models. Cannot be used with warming levels, and valid start and end years must be between 1981 and 2019. |

---

</details>

In [ ]:
xmy = shock_XMY(
    ## Select your extreme type: "hot", "cold"
    extreme = "hot",

    ## Location -- uncomment your desired option
    station_name = "Sacramento Executive Airport (KSAC)",
    # latitude = latitude,
    # longitude = longitude,

    ## Approach -- uncomment your desired option
    warming_level = 2.0,
    # start_year = start_year,  
    # end_year = end_year, 

    ## Uncomment to use ERA5 reanalysis, valid only with time-based approach with start and end years between 1981 and 2019
    # reanalysis = True,
    verbose = True,
)

xmy.generate_xmy()

**Export to non-EPW format.** Shock XMY profiles are exported in `.epw` format by default, but can be exported as `.csv` file formats using the method `export_xmy_data` with the argument `extension="your_choice"` as shown below.

In [ ]:
xmy.export_xmy_data(extension="csv")

#### Generate a Persistence Extreme Meteorological Year (persistence XMY)

<details>
<summary><strong><span style="color: #1577b4;">▶ Click Here for the persistence XMY Customization Guide</span></strong></summary>

---

The persistence XMY methodology utilizes the following changes to the TMY methodology to capture extremes at the annual scale:

1. Uses hourly air temperature, reflecting utility priorities when using XMYs.
2. Constructs the final XMY by selecting a year for each hour - rather than a year for each month - of the final representative year.
3. Selects each year using percentiles, rather than distance from a climatological CDF. This, essentially, applies the Cal-Adapt Standard Year methodology to persistence XMY construction.


|**Argument**|**Options**|**Notes**|
|------------|-----------|-----------------------|
|q | float between 0 and 1 | Extreme quantile
|Location: station_name|HadISD weather stations | Check out the <span style="color: blue;\">[available station list](https://cadcat.s3.amazonaws.com/hadisd/hadisd_stations.csv)</span> for station options. Cached areas include counties, demand forecast zones, or service territories.|
|Location: latitude, longitude|Custom lat-lon pairs| Note: there is a small buffer applied to lat-lon arguments to retrieve the closest gridcell. |
|approach|"Warming Level", "Time"|Default = "Warming Level"|
|warming_level|Valid range is 0.5 - 4.0°C|Fully customizeable GWL. Default GWL = 1.2°C |
|start_year|Valid range is 1980-2099. | We recommend a 30-year period such as 1990-2020. |
|end_year|Valid range is 1980-2099.  | We recommend a 30-year period such as 1990-2020. |
|reanalysis|True, False  | True to use ERA5 reanalysis instead of models. Cannot be used with warming levels, and valid start and end years must be between 1981 and 2019. |

---

</details>

In [ ]:
xmy = persistence_XMY(
    ## Select your persistence quantile, range 0-1
    q = 0.5,

    ## Location -- uncomment your desired option
    station_name = "Sacramento Executive Airport (KSAC)",
    # latitude = latitude,
    # longitude = longitude,

    ## Approach -- uncomment your desired option
    warming_level = 2.0,
    # start_year = start_year,
    # end_year = end_year,

    ## Uncomment to use ERA5 reanalysis, valid only with time-based approach with start and end years between 1981 and 2019
    # reanalysis = True,
    verbose = True,
)

xmy.generate_xmy()

**Export to non-EPW format.** Persistence XMY profiles are exported in `.epw` format by default, but can be exported as `.csv` file formats using the method `export_xmy_data` with the argument `extension="your_choice"` as shown below.

In [ ]:
xmy.export_xmy_data(extension="csv")